# 03 — Neuron + Head Pruning (Wanda)

Magnitude × activation pruning of MLP neurons and attention heads. Single forward pass; recover briefly with QLoRA.

**Papers**
- Wanda https://arxiv.org/abs/2306.11695
- horseee/LLaMA-Pruning (GitHub)

In [ ]:
# ###### Colab bootstrap ######
# On Colab the [experiments] extras pulls both requirements_package.txt and
# requirements_experiments.txt in one pip command — single source of truth lives
# in setup.py's extras_require. Then bootstrap() loads Colab Secrets into
# os.environ and brings up Tailscale so *.ts.net hostnames are reachable.
# Locally bootstrap() just loads .env and checks the tailnet — no installs.
#
# Required Colab Secrets (key icon → Add new secret → toggle "Notebook access"):
#   TAILSCALE_AUTHKEY   – from https://login.tailscale.com/admin/settings/keys
#   MLFLOW_TRACKING_URI, MLFLOW_EXPERIMENT_NAME, MLFLOW_TRACKING_INSECURE_TLS
#   MINIO_ENDPOINT / MINIO_ACCESS_KEY / MINIO_SECRET_KEY / MINIO_SECURE
#   HF_TOKEN
import os, subprocess, sys
IN_COLAB = "COLAB_GPU" in os.environ or "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "burnit_bg[experiments] @ git+https://github.com/kirilyotov/BurnIT-BG.git",
    ])

from utils.colab import bootstrap
bootstrap()


In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"Device: {props.name}")
    print(f"VRAM:   {props.total_memory / 1024**3:.2f} GB")
    print(f"Compute capability: {props.major}.{props.minor}")


## 1. Setup & config

In [ ]:
import sys, os
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'data_platform').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from data_platform.common import set_env
from data_platform.storage import MinioStorage

from experiments.shared.mlflow_utils import setup_run, log_responses, stage, log_training_curves
from experiments.shared.inference_utils import (
    run_full_test_battery,
    TEST_PROMPTS_IN_DOMAIN, TEST_PROMPTS_OUT_OF_DOMAIN, TEST_PROMPTS_EDGE,
    format_prompt,
)
from experiments.shared.model_utils import (
    DEFAULT_MODEL_NAME, load_model_unsloth, apply_qlora, count_trainable_params,
    cuda_device_info,
)
from experiments.shared.eval_utils import compute_perplexity, benchmark_speed, vram_snapshot
from experiments.shared.dataset_utils import (
    load_alpaca_dataset, dataset_statistics, alpaca_to_prompt,
)
from experiments.shared.dataset_browser import list_datasets, pick_dataset, download_dataset, resolve


In [ ]:
set_env(quiet=True)
tracking, tags, run_name = setup_run(
    experiment='neuron_pruning',
    model=DEFAULT_MODEL_NAME,
    stage="experiment",
    mlflow_experiment_name='burnit-bg-experiments',
)
print(f"run_name = {run_name}")
print(f"tags     = {tags}")
print(f"machine  = {cuda_device_info()}")


## 2. Data loading

In [ ]:
# ###### Pick a dataset from MinIO (or fall back to local data_prep/) ######
# Set DATASET_PREFIX in .env / Colab secrets to skip the picker, e.g.
#   DATASET_PREFIX=data_prep/processed/mental-health
# Or pass `auto=True` below for non-interactive runs.

DEFAULT_PREFIX = os.getenv("DATASET_PREFIX", "data_prep/processed")
try:
    local_dataset_dir = resolve(prefix=DEFAULT_PREFIX, auto=False)
    TRAIN_PATH = local_dataset_dir / "train.jsonl"
    EVAL_PATH  = local_dataset_dir / "eval.jsonl"
except (FileNotFoundError, Exception) as exc:
    print(f"[data] MinIO lookup failed ({exc}); falling back to local data_prep/processed/")
    TRAIN_PATH = REPO_ROOT / "data_prep/processed/train.jsonl"
    EVAL_PATH  = REPO_ROOT / "data_prep/processed/eval.jsonl"

train_records = list(load_alpaca_dataset(TRAIN_PATH))
eval_records  = list(load_alpaca_dataset(EVAL_PATH))
train_stats = dataset_statistics(train_records)
eval_stats  = dataset_statistics(eval_records)
print(f"train: {len(train_records)} rows  eval: {len(eval_records)} rows")
print(train_stats)


## 2b. Wanda calibration + neuron scoring

In [ ]:
# Wanda: prune the neurons with the lowest |W| * ||X||_2 score.
import torch, torch.nn as nn

CALIBRATION_SAMPLES = 128
calibration_text = [alpaca_to_prompt(r) for r in train_records[:CALIBRATION_SAMPLES]]

@torch.no_grad()
def collect_activation_norms(model, tokenizer, texts):
    norms = {}  # name -> running sum of ||X||_2 per input channel
    handles = []
    def make_hook(name):
        def _hook(_m, inputs, _output):
            x = inputs[0]
            n = x.float().pow(2).sum(dim=(0, 1)).sqrt()  # per-input-channel L2
            norms[name] = norms.get(name, 0) + n.detach().cpu()
        return _hook
    for name, mod in model.named_modules():
        if isinstance(mod, nn.Linear) and any(k in name for k in ("mlp.up_proj", "mlp.gate_proj")):
            handles.append(mod.register_forward_hook(make_hook(name)))
    try:
        for txt in texts:
            ids = tokenizer(txt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
            model(**ids)
    finally:
        for h in handles: h.remove()
    return norms

activation_norms = collect_activation_norms(model, tokenizer, calibration_text)
print(f"collected activations for {len(activation_norms)} MLP projections")


## 3. Prune lowest-scoring neurons

In [ ]:
# For each MLP layer pair (up_proj, gate_proj), score weight rows by
# |W| * ||X||, then drop the lowest 30% by zeroing them out.
SPARSITY = 0.30
import torch
removed_total = 0
for name, mod in model.named_modules():
    if not (isinstance(mod, nn.Linear) and any(k in name for k in ("mlp.up_proj", "mlp.gate_proj"))):
        continue
    activations = activation_norms.get(name)
    if activations is None: continue
    score = mod.weight.detach().abs() * activations.to(mod.weight.device)
    per_neuron = score.sum(dim=1)
    threshold = torch.quantile(per_neuron, SPARSITY)
    keep_mask = per_neuron > threshold
    mod.weight.data[~keep_mask] = 0.0  # soft prune (no shape change)
    removed_total += int((~keep_mask).sum())

tracking.log_params({"prune.sparsity": SPARSITY})
tracking.log_metrics({"prune.neurons_zeroed": float(removed_total)})
print(f"zeroed {removed_total} MLP neurons across all layers")


## 4. Recovery fine-tune (lm_head + last 3 blocks)

In [ ]:
# Freeze everything except the language-model head and the last 3
# decoder blocks; cheap recovery with QLoRA.
for p in model.parameters(): p.requires_grad_(False)
last_three = list(model.model.layers)[-3:]
for layer in last_three:
    for p in layer.parameters(): p.requires_grad_(True)
if hasattr(model, "lm_head"):
    for p in model.lm_head.parameters(): p.requires_grad_(True)

from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

train_ds = Dataset.from_list(train_records).map(lambda r: {**r, "text": alpaca_to_prompt(r, eos_token=tokenizer.eos_token)})
eval_ds  = Dataset.from_list(eval_records).map(lambda r: {**r, "text": alpaca_to_prompt(r, eos_token=tokenizer.eos_token)})

OUTPUT_DIR = REPO_ROOT / "tmp/experiments/neuron_pruning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR), num_train_epochs=2,
    per_device_train_batch_size=2, gradient_accumulation_steps=4,
    learning_rate=1e-4, warmup_ratio=0.03, logging_steps=10,
    save_strategy="epoch", bf16=True, optim="adamw_8bit",
    report_to=[], lr_scheduler_type="cosine", max_grad_norm=1.0,
)
with tracking.run(run_name=run_name, tags=tags):
    with stage(tracking, "recovery_train"):
        trainer = SFTTrainer(
            model=model, tokenizer=tokenizer,
            train_dataset=train_ds, eval_dataset=eval_ds,
            dataset_text_field="text", max_seq_length=2048,
            args=training_args,
        )
        trainer.train()


## 5. Evaluation

In [ ]:
with stage(tracking, "evaluate"):
    ppl = compute_perplexity(model, tokenizer, [r["output"] for r in eval_records[:64]])
    speed = benchmark_speed(model, tokenizer, new_tokens=128, runs=3)
    tracking.log_metrics({
        "eval_perplexity": float(ppl),
        "tokens_per_sec": speed["tokens_per_sec_mean"],
        **{f"vram.{k}": v for k, v in vram_snapshot().items()},
    })
    print(f"after pruning: ppl={ppl:.3f}  tps={speed['tokens_per_sec_mean']:.1f}")


## 6. Inference test

In [ ]:
# ###### Inference test (mental-health prompts) ######
with stage(tracking, "inference_test"):
    batteries = run_full_test_battery((model, tokenizer), max_new_tokens=256)
    log_responses(
        tracking,
        experiment='neuron_pruning',
        model_checkpoint=str(REPO_ROOT / "tmp/experiments/neuron_pruning"),
        **batteries,
    )
    for k, v in batteries.items():
        print(f"-- {k} --")
        for entry in v[:2]:
            print(f"  Q: {entry['prompt'][:80]}\n  A: {entry['response'][:200]}\n")


## 7. Save via MLflow + GGUF export

In [ ]:
# ###### Save model via MLflow (single source of truth) ######
# tracking.log_model logs the model artifact under runs:/<id>/model and,
# when registered_model_name is set, adds a new version to the Models tab.
# MLflow's artifact store backs onto MinIO — no separate upload needed.
with stage(tracking, "save_model"):
    try:
        tracking.log_model(
            model,
            flavor="transformers",
            artifact_path="model",
            registered_model_name='burnit-bg-neuron-pruning',
            input_example=None,
        )
        print("[save] model logged via MLflow")
    except Exception as exc:
        print(f"[save] log_model failed: {exc}")

# Optional: GGUF export for offline local inference (RTX 3050 / Ollama).
# The GGUF is added as a run artifact under `gguf/`.
with stage(tracking, "gguf_export"):
    try:
        from experiments.shared.model_utils import export_gguf
        gguf_path = export_gguf(model, tokenizer, REPO_ROOT / "tmp/experiments/neuron_pruning/gguf", quantization="q4_k_m")
        tracking.save_data(gguf_path, artifact_path="gguf")
        print(f"[save] GGUF logged: {gguf_path}")
    except Exception as exc:
        print(f"[save] GGUF export skipped: {exc}")
